# Stage 7A — A safe prediction function

**Goal:** Load the saved model and predict from only three creation-time fields: `complaint_type`, `borough`, and `created_date`.

**Why:** Serving requires a clear input contract. The function must reject missing, extra, or invalid fields and return a model version with every prediction.

This is a local demonstration function. The probability is an estimate; we have not checked calibration or approved the model for operational use. Stage 7B will wrap it in a local API.

In [ ]:
# Cell 2 — Why: load only our hash-verified model and matching environment.
from google.colab import drive
from pathlib import Path
import hashlib, json, joblib, platform, pandas as pd, numpy as np, sklearn

drive.mount("/content/drive")
root = Path("/content/drive/MyDrive/nyc311_mlops")
run_id = "20260926T120702352046Z"
run_dir = root / "runs" / run_id
record = json.loads((run_dir / "run.json").read_text())
model_file = run_dir / "model.joblib"
if record["run_id"] != run_id or hashlib.sha256(model_file.read_bytes()).hexdigest() != record["model_sha256"]:
    raise ValueError("Wrong run or changed model")
versions = {"python": platform.python_version(), "pandas": pd.__version__,
            "numpy": np.__version__, "scikit_learn": sklearn.__version__,
            "joblib": joblib.__version__}
if versions != record["versions"]:
    raise RuntimeError("Model and runtime versions differ; use the original environment")
model = joblib.load(model_file)
print("Loaded model version:", run_id)

In [ ]:
# Cell 3 — Why: define exactly what a new request may contain.
from datetime import datetime

allowed = {"complaint_type", "borough", "created_date"}
boroughs = {"BRONX", "BROOKLYN", "MANHATTAN", "QUEENS", "STATEN ISLAND", "Unspecified"}

def check_request(item):
    if not isinstance(item, dict) or set(item) != allowed:
        raise ValueError(f"Supply exactly these fields: {sorted(allowed)}")
    for field in ["complaint_type", "borough"]:
        if not isinstance(item[field], str) or not item[field].strip():
            raise ValueError(f"{field} must be nonempty text")
    if item["borough"] not in boroughs:
        raise ValueError("borough is not a recognized NYC borough")
    if not isinstance(item["created_date"], str):
        raise ValueError("created_date must be local NYC date-time text")
    try:
        moment = datetime.fromisoformat(item["created_date"])
        if moment.tzinfo is not None:
            raise ValueError("Use NYC local time without a timezone suffix")
        pd.Timestamp(moment).tz_localize("America/New_York", ambiguous="raise", nonexistent="raise")
    except Exception as error:
        raise ValueError(f"Invalid NYC created_date: {error}") from error
    return moment

print("Input contract:", sorted(allowed))

In [ ]:
# Cell 4 — Why: derive the same time features used during training.
def make_features(item, moment):
    row = {"complaint_type": item["complaint_type"], "borough": item["borough"],
           "month": moment.month, "hour": moment.hour, "weekday": moment.weekday()}
    return pd.DataFrame([row], columns=record["features"])

example = {"complaint_type": "HEAT/HOT WATER", "borough": "BRONX",
           "created_date": "2026-09-26T10:00:00.000"}
example_features = make_features(example, check_request(example))
print(example_features.to_dict(orient="records")[0])

In [ ]:
# Cell 5 — Why: return a probability, a decision, and model lineage.
known_types = set(model.named_steps["prepare"].named_transformers_["text"]
                  .named_steps["encode"].categories_[0])

def predict_request(item):
    moment = check_request(item)
    probability = float(model.predict_proba(make_features(item, moment))[0, 1])
    threshold = float(record["decision_threshold"])
    return {"probability_late": round(probability, 4),
            "predicted_label": int(probability >= threshold),
            "threshold": threshold, "model_run_id": run_id,
            "new_complaint_type": item["complaint_type"] not in known_types}

print(predict_request(example))

In [ ]:
# Cell 6 — Why: test that leaked or broken inputs are rejected.
bad_examples = [
    {**example, "closed_date": "2026-09-27T10:00:00.000"},
    {"complaint_type": "HEAT/HOT WATER", "created_date": example["created_date"]},
    {**example, "created_date": "not a date"},
]
for item in bad_examples:
    try:
        predict_request(item)
        raise AssertionError("Bad input was accepted")
    except ValueError as error:
        print("Rejected:", str(error).split(":")[0])

In [ ]:
# Cell 7 — Why: save the contract for the API stage.
contract = {
    "contract_version": "stage7a-v1", "model_run_id": run_id,
    "required_fields": sorted(allowed),
    "created_date_rule": "NYC local ISO date-time, without timezone suffix",
    "positive_label": "not closed within 72 hours",
    "threshold": record["decision_threshold"],
    "example_request": example, "example_response": predict_request(example)
}
contract_dir = root / "contracts" / run_id
contract_dir.mkdir(parents=True, exist_ok=True)
contract_file = contract_dir / "stage7a-v1.json"
contract_file.write_text(json.dumps(contract, indent=2) + "\n", encoding="utf-8")
print("Saved contract:", contract_file)

# Stage 7A takeaway and quiz

**What you learned:** A model is useful only if new requests can be checked and transformed the same way as training data. A response includes the model version so we know which model produced it. An unfamiliar complaint type is flagged for monitoring.

**Easy quiz:**

1. Why does the input reject `closed_date`?
2. Where do `month`, `hour`, and `weekday` come from for a new request?
3. Why does the response include `model_run_id`?
4. If `new_complaint_type` is true, should we quietly assume the prediction is reliable?

Send me the output of Cells 5–7 and your answers. Stage 7B will expose this function through a local API.